# CESM2-LE Niño3.4 Index — Exploration

Plots the **Niño3.4 ENSO index** for a selected CESM2-LE ensemble member.

Niño3.4 = area-mean SST anomaly over (5°S–5°N, 170°W–120°W), 5-month smoothed and normalised.  
Phases: El Niño (+1), Neutral (0), La Niña (−1), threshold = 0.4 σ, minimum 6 consecutive months.

**Plots produced:**
1. Full time series — all months, all years
2. Time series for a selected calendar month only
3. SST composite maps for El Niño, Neutral, and La Niña phases

In [ ]:
import sys
from pathlib import Path
import numpy as np
import xarray as xr
import netCDF4 as nc
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
try:
    import cmocean
    CMAP_SST = cmocean.cm.balance
except ImportError:
    CMAP_SST = 'RdBu_r'

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))
from configs import paths

MONTH_LABELS = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
MONTH_NAMES  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

## 1. User Parameters
Edit the cell below before running the notebook.

In [ ]:
# ─────────────────────────────────────────────────
# USER PARAMETERS
# ─────────────────────────────────────────────────

ens_member  = 0     # Ensemble member index (0–99)
                    #   0–49  → CMIP6 members
                    #  50–99  → SMBB members

plot_month  = 9     # Calendar month for the monthly time-series plot
                    # (1 = Jan, 9 = Sep, 12 = Dec)

# Composite map settings
clim_vmin, clim_vmax = -0.5, 0.5    # SST anomaly colorbar limits (K)

# File locations
NINO34_FILE = paths.CESM2LE_DIR / 'climate_indices' / 'cesm2le_nino34_index.nc'
SST_DIR     = paths.CESM2LE_SST_DIR / 'mon'
GRID_FILE   = paths.CESM2LE_SST_DIR / 'grid' / 'cesm2le_sst_grid.nc'

In [ ]:
import matplotlib.patches as mpatches

# ── Niño3.4 region box ──────────────────────────────────────────────────────
#   5°S–5°N, 170°W–120°W  =  190–240°E
NINO34_BOX = dict(latmin=-5, latmax=5, lonmin=190, lonmax=240)

proj  = ccrs.PlateCarree(central_longitude=180)
trans = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=(12, 4), subplot_kw={'projection': proj})
ax.set_global()
ax.coastlines(linewidth=0.6, color='0.4')
ax.set_facecolor('white')

b = NINO34_BOX
rect = mpatches.Rectangle(
    (b['lonmin'], b['latmin']),
    b['lonmax'] - b['lonmin'],
    b['latmax'] - b['latmin'],
    linewidth=2, edgecolor='tab:red', facecolor='tab:red',
    alpha=0.35, transform=trans, zorder=3
)
ax.add_patch(rect)

gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

ax.legend(
    handles=[mpatches.Patch(facecolor='tab:red', alpha=0.5,
                            label='Niño3.4  (5°S–5°N, 170°W–120°W)')],
    loc='lower left', fontsize=9, framealpha=0.9
)
ax.set_title('Niño3.4 index box', fontsize=12, pad=8)
fig.tight_layout()
plt.show()

## 2. Load Niño3.4 Data

In [ ]:
ds_n34 = xr.open_dataset(NINO34_FILE)
print(ds_n34)

years = ds_n34['nyr'].values          # (nyear,)
nyear = len(years)

# Full ensemble arrays — shape (nens, nyear, 12)
nino34_all = ds_n34['nino34_months'].values
labels_all = ds_n34['nino_labels'].values

# Select chosen ensemble member
nino34 = nino34_all[ens_member]       # (nyear, 12)
labels = labels_all[ens_member]       # (nyear, 12)

# Flatten to 1-D monthly time series
nino34_flat = nino34.reshape(-1)      # (ntime,)
labels_flat = labels.reshape(-1)

# Build datetime index
dates = pd.date_range(start=f'{years[0]}-01', periods=nyear * 12, freq='MS')

print(f'\nEnsemble member  : {ens_member}')
print(f'Years            : {years[0]}–{years[-1]}  ({nyear} years, {len(dates)} months)')
print(f'El Niño months   : {(labels_flat == 1).sum()}')
print(f'La Niña months   : {(labels_flat == -1).sum()}')
print(f'Neutral months   : {(labels_flat == 0).sum()}')

## 3. Time Series — All Months

Full monthly Niño3.4 index with El Niño (red) and La Niña (blue) phase shading.

In [ ]:
def shade_phases(ax, dates, phases, alpha=0.2):
    """Shade background red (El Niño) and blue (La Niña) by phase."""
    start = 0
    for i in range(1, len(phases)):
        if phases[i] != phases[start]:
            if phases[start] != 0:
                color = 'red' if phases[start] == 1 else 'blue'
                ax.axvspan(dates[start], dates[i - 1], color=color, alpha=alpha)
            start = i
    if phases[start] != 0:
        color = 'red' if phases[start] == 1 else 'blue'
        ax.axvspan(dates[start], dates[-1], color=color, alpha=alpha)

fig, ax = plt.subplots(figsize=(15, 4))

shade_phases(ax, dates, labels_flat)
ax.plot(dates, nino34_flat, color='black', linewidth=0.8)
ax.axhline(0,    color='black', linewidth=0.6, linestyle='--', alpha=0.4)
ax.axhline( 0.4, color='red',   linewidth=0.6, linestyle=':',  alpha=0.6, label='+0.4 σ')
ax.axhline(-0.4, color='blue',  linewidth=0.6, linestyle=':',  alpha=0.6, label='−0.4 σ')

ax.set_ylabel('Niño3.4 index (normalised)', fontsize=11)
ax.set_xlabel('Year', fontsize=11)
ax.set_title(f'CESM2-LE Niño3.4 — Member {ens_member}  ({years[0]}–{years[-1]})',
             fontsize=13)
ax.legend(fontsize=9, loc='upper right')
ax.tick_params(labelsize=10)
fig.tight_layout()
plt.show()

## 4. Time Series — Selected Month

Niño3.4 index for a single calendar month across all years.  
Bars are coloured by phase: red = El Niño, blue = La Niña, grey = neutral.

In [ ]:
m_idx    = plot_month - 1   # 0-based
mon_name = MONTH_NAMES[m_idx]

def phase_colors(phases):
    return ['red' if p == 1 else 'blue' if p == -1 else 'lightgray' for p in phases]

fig, ax = plt.subplots(figsize=(14, 4))

cols = phase_colors(labels[:, m_idx])
ax.bar(years, nino34[:, m_idx], color=cols, alpha=0.85, width=0.85, edgecolor='none')
ax.axhline(0,    color='black', linewidth=0.7)
ax.axhline( 0.4, color='red',   linewidth=0.6, linestyle=':', alpha=0.7)
ax.axhline(-0.4, color='blue',  linewidth=0.6, linestyle=':', alpha=0.7)

ax.set_ylabel('Niño3.4 index (normalised)', fontsize=11)
ax.set_xlabel('Year', fontsize=11)
ax.set_title(f'CESM2-LE Niño3.4 — {mon_name} — Member {ens_member}', fontsize=13)
ax.tick_params(labelsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# ── Load 2D lat/lon from grid file (TLAT / TLONG) ───────────────────────────
ds_grid = nc.Dataset(str(GRID_FILE), 'r')
ds_grid.set_auto_mask(False)
lat_grid = np.array(ds_grid.variables['TLAT'][:],  dtype=np.float64)   # (nj, ni)
lon_grid = np.array(ds_grid.variables['TLONG'][:], dtype=np.float64)   # (nj, ni)
ds_grid.close()
print(f'lat_grid shape : {lat_grid.shape}   range: [{lat_grid.min():.1f}, {lat_grid.max():.1f}]')
print(f'lon_grid shape : {lon_grid.shape}   range: [{lon_grid.min():.1f}, {lon_grid.max():.1f}]')


In [ ]:
def load_member_sst(member_idx, sst_dir):
    """
    Load SST for one ensemble member across all 12 calendar months.
    Returns sst (ntime, nlat, nlon) in chronological order.
    Members 0–49 → first50members files; 50–99 → last50members files.
    """
    if member_idx < 50:
        group, grp_idx = 'first50', member_idx
    else:
        group, grp_idx = 'last50', member_idx - 50

    monthly = []
    for m_label in MONTH_LABELS:
        fpath = sst_dir / f'sst_cesmle_{group}members_mon_{m_label}_199001-210012.nc'
        ds    = nc.Dataset(str(fpath), 'r')
        ds.set_auto_mask(False)
        sst_m = ds.variables['sst_mon'][grp_idx].astype(np.float32)  # (nyear, nlat, nlon)
        ds.close()
        monthly.append(sst_m)
        print(f'  Loaded {m_label}', end='\r')

    # Stack: (nyear, 12, nlat, nlon) → reshape → (ntime, nlat, nlon) chronologically
    sst_stack = np.stack(monthly, axis=1)
    ny, _, nlat, nlon = sst_stack.shape
    sst = sst_stack.reshape(ny * 12, nlat, nlon)
    print(f'  SST loaded: shape {sst.shape}')
    return sst


print(f'Loading SST for member {ens_member}...')
sst = load_member_sst(ens_member, SST_DIR)


In [ ]:
sst_clim = np.nanmean(sst, axis=0, keepdims=True)
sst_anom = sst - sst_clim   # (ntime, nlat, nlon)

# ── Build SST composites ──────────────────────────────────────────────────────
# labels_flat shape: (ntime,)  — +1 El Niño, 0 Neutral, -1 La Niña
mask_nino = labels_flat == 1
mask_neut = labels_flat == 0
mask_nina = labels_flat == -1

n_nino = int(mask_nino.sum())
n_neut = int(mask_neut.sum())
n_nina = int(mask_nina.sum())

sst_nino = np.nanmean(sst_anom[mask_nino], axis=0)   # (nlat, nlon)
sst_neut = np.nanmean(sst_anom[mask_neut], axis=0)
sst_nina = np.nanmean(sst_anom[mask_nina], axis=0)

print(f'El Niño months  : {n_nino}')
print(f'Neutral months  : {n_neut}')
print(f'La Niña months  : {n_nina}')
print(f'Composite shape : {sst_nino.shape}')


In [ ]:
proj = ccrs.PlateCarree(central_longitude=180)

fig, axes = plt.subplots(1, 3, figsize=(22, 6), subplot_kw={'projection': proj})

for ax, data, title in [
    (axes[0], sst_nino, f'El Niño  (n = {n_nino})'),
    (axes[1], sst_neut, f'Neutral  (n = {n_neut})'),
    (axes[2], sst_nina, f'La Niña  (n = {n_nina})'),
]:
    ax.set_global()
    ax.coastlines(linewidth=0.5, color='k')
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=2)

    im = ax.pcolormesh(lon_grid, lat_grid, data,
                       cmap=CMAP_SST,
                       vmin=clim_vmin, vmax=clim_vmax,
                       shading='auto',
                       transform=ccrs.PlateCarree())

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(title, fontsize=12, pad=6)

fig.subplots_adjust(bottom=0.12)
cbar_ax = fig.add_axes([0.25, 0.06, 0.5, 0.03])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label('SST Anomaly (K)', fontsize=11)

fig.suptitle(
    f'SST composites — Niño3.4 ENSO phases\n'
    f'CESM2-LE member {ens_member},  {years[0]}–{years[-1]}',
    fontsize=13
)
plt.show()